In [ ]:
#S_A=P_A-1を確認中

In [60]:
from __future__ import print_function, division
import sys,os
# line 4 and line 5 below are for development purposes and ca be removed
qspin_path = os.path.join(os.getcwd(),"../../")
sys.path.insert(0,qspin_path)
from quspin.operators import hamiltonian # Hamiltonians and operators
#from quspin.basis import spinless_fermion_basis_1d # Hilbert space fermion basis
from quspin.basis import spin_basis_1d # Hilbert space spin basis
import numpy as np # generic math functions
import matplotlib.pyplot as plt # plotting library
from scipy.linalg import expm, sinm, cosm
from numpy.linalg import multi_dot
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy.sparse.linalg import inv, eigs
from scipy.linalg import svdvals
from scipy.stats import unitary_group
from scipy import linalg
from numpy import linalg as LA
import pylab
import time
import numba
from numba import njit,c16,i8,u2
import sys
import math

np.set_printoptions(precision=2)
np.set_printoptions(suppress=True)
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

time_sta = time.time()

####################
#cilyinder geometry#
####################

def idx(x,y,Lx,Ly):
    return (x % Lx) + Lx * (y % Ly)
def inv_idx(i,Lx,Ly):
    return i % Lx, i // Lx

def dir_x(x,y,Lx,Lv):
    return x+Lx*y
def dir_y(x,y,Lx,Lv):
    return x+Lx*y+Lv
def dir_xy(x,y,Lx,Lv):
    return x+Lx*y+2*Lv


def create_transformation_pv_link(Lv, Lx, Ly):
    #############################################
    ############ plaquette part #################
    #############################################
    p_indd = []
    for ip in range(Lx * Ly):
        ix, iy = ip % Lx, ip // Lx
        ipx=(ix + 1) % Lx
        ipy=(iy + 1) % Ly
        p_indd.append([
            dir_x(ix, iy, Lx, Lv)+2*Lv,
            dir_y(ix, iy, Lx, Lv)+2*Lv,
            dir_x(ix, ipy, Lx, Lv)+2*Lv,
            dir_y(ipx, iy, Lx, Lv)+2*Lv
        ])
    #print(p_indd)
    ############################################
    ############ vertex part ###################
    ############################################
    v_indd = []
    for iv in range(Lx * Ly):
        ix, iy = iv % Lx, iv // Lx
        imx=(ix - 1) % Lx
        imy=(iy - 1) % Ly
        v_indd.append([
            dir_x(ix, iy, Lx, Lv),
            dir_y(ix, iy, Lx, Lv),
            dir_x(imx, iy, Lx, Lv),
            dir_y(ix, imy, Lx, Lv),
        ])
    #print(v_indd)
    return p_indd, v_indd


#################################################################
def initial_stabilizer_state(Ld,Lxd,Lyd,p_ind,v_ind):
    # Ld: total # of qubits
    MRi=np.zeros((Ld,2*Ld+1),dtype='uint32')
### star stabilizers  XXXX ###
    for iv in range(Lxd*Lyd-1):
        #ivs=iv+(Lxd*(Lyd)-1)  # note that iv and ivs are different.
        MRi[iv][v_ind[iv][0]]=1 # neglect a phase factor
        MRi[iv][v_ind[iv][1]]=1
        MRi[iv][v_ind[iv][2]]=1
        MRi[iv][v_ind[iv][3]]=1
        #独立ではない最後のAvだけ取らない   
### plaquette stabilizers ZZZZ ###
    for ip in range(Lxd*Lyd-1):
        #MRi[Lxd*(Lyd)-1+ip][p_ind[ip]]=1 # neglect a phase factor
        MRi[ip+Lxd*Lyd-1][p_ind[ip][0]]=1 # neglect a phase factor
        MRi[ip+Lxd*Lyd-1][p_ind[ip][1]]=1
        MRi[ip+Lxd*Lyd-1][p_ind[ip][2]]=1
        MRi[ip+Lxd*Lyd-1][p_ind[ip][3]]=1        
        #独立ではない最後のBpだけ取らない
### Add logical operator prod X on upper rough boundary ###
    for kk in range(Lxd):
        MRi[Ld-2][kk]=1 #prod X (x-direction nonctractable)
        #print(kk,kk)
    for kk in range(Lyd):
        lo=Lxd+(2*Lxd)*kk
        MRi[Ld-1][lo]=1 #prod X (y-direction nonctractable)
        #print(kk,lo)
    #for kk in range(Ld):
    #    print(kk,MRi[kk,0:12],MRi[kk,12:24])
    #sys.exit()
    return MRi


#################################################
def create_partition_TEE2(Ld,Lx,Ly,p_indd,v_indd):
#############################################
############ plaquette part #################
#############################################
    dhw=1
    Lv=Ly-3
    A_indd=set()
    for iy in range(2,Ly-1,1):
        for ix in range(0,dhw,1):
            ip=ix+iy*Lx
            #ip=idx(ix,iy,Lx,Ly)
            A_indd.update(p_indd[ip])
            subtracted = [x - 2*Ld for x in p_indd[ip]]
            print("A:ix=",ix,"iy=",iy,"ip=",ip,subtracted)
    C_indd=set()
    for iy in range(2+Lv//2+1,Ly-1,1):
        for ix in range(dhw,Lx-2,1):
            ip=ix+iy*Lx
            C_indd.update(p_indd[ip])
            subtracted = [x - 2*Ld for x in p_indd[ip]]
            print("C:ix=",ix,"iy=",iy,"ip=",ip,subtracted)
    C_indd=C_indd-A_indd

    B_indd=set()
    for iy in range(2,2+Lv//2+1,1):
        for ix in range(dhw,Lx-2,1):
            ip=ix+iy*Lx
            B_indd.update(p_indd[ip])
            subtracted = [x - 2*Ld for x in p_indd[ip]]
            print("B:ix=",ix,"iy=",iy,"ip=",ip,subtracted)           
    B_indd=B_indd-A_indd-C_indd
        
    C_indd=C_indd-A_indd-B_indd## elimination of elements included in A_indd and B_indd from C_indd
    AB_indd=A_indd.union(B_indd)
    AC_indd=A_indd.union(C_indd)
    BC_indd=B_indd.union(C_indd)
    ABC_indd=A_indd.union(B_indd,C_indd)
    A_indd=list(A_indd)
    A_indd+=[x-2*Ld for x in A_indd]
    B_indd=list(B_indd)
    B_indd+=[x-2*Ld for x in B_indd]
    C_indd=list(C_indd)
    C_indd+=[x-2*Ld for x in C_indd]
    AB_indd=list(AB_indd)
    AB_indd+=[x-2*Ld for x in AB_indd]
    BC_indd=list(BC_indd)
    BC_indd+=[x-2*Ld for x in BC_indd]
    AC_indd=list(AC_indd)
    AC_indd+=[x-2*Ld for x in AC_indd]
    ABC_indd=list(ABC_indd)
    ABC_indd+=[x-2*Ld for x in ABC_indd]

    print("TEE2")
    print("A_indd=",sorted(A_indd))
    print("B_indd=",sorted(B_indd))
    print("C_indd=",sorted(C_indd))
    #print("AB_indd=",sorted(AB_indd))
    #print("AC_indd=",sorted(AC_indd))
    #print("BC_indd=",sorted(BC_indd))
    #print("ABC_indd=",sorted(ABC_indd))
    
    return A_indd, B_indd, C_indd,AB_indd, AC_indd, BC_indd,ABC_indd 


def TEE_cal(MRFd,A_indd,B_indd,C_indd,AB_indd,AC_indd,BC_indd,ABC_indd):
    # partition AC#
    rankAC=rank_mod2_v3(MRFd[:,AC_indd].copy())
    SAC=rankAC-(len(AC_indd)//2)
######################
    rankAB=rank_mod2_v3(MRFd[:,AB_indd].copy())
    SAB=rankAB-(len(AB_indd)//2)
######################
    rankBC=rank_mod2_v3(MRFd[:,BC_indd].copy())
    SBC=rankBC-(len(BC_indd)//2)
######################
    rankABC=rank_mod2_v3(MRFd[:,ABC_indd].copy())
    SABC=rankABC-(len(ABC_indd)//2)
######################
    rankC=rank_mod2_v3(MRFd[:,C_indd].copy())
    SC=rankC-(len(C_indd)//2)
    rankB=rank_mod2_v3(MRFd[:,B_indd].copy())
    SB=rankB-(len(B_indd)//2)
    rankA=rank_mod2_v3(MRFd[:,A_indd].copy())
    SA=rankA-(len(A_indd)//2)
    #print(A_indd)
    #print(rankA,len(A_indd)//2)
    #print(MRFd[:,A_indd])
    #print(B_indd)
    #print(rankB,len(B_indd)//2)
    #print(MRFd[:,B_indd])
    TEE=SA+SB+SC-SAC-SBC-SAB+SABC
    #print("SA=",SA,"SB=",SB)
    print("SA=",SA,"SB=",SB,"SC=",SC,"SAC=",SAC,"SBC=",SBC,"SAB=",SAB,"SABC=",SABC)
    print("TEE=",TEE)
    return TEE

#######################################################################
def rank_mod2_v3(MRdd):
    # MRdA construct
    i = 0
    Ic = MRdd.shape[0]
    Jc = MRdd.shape[1]
    for j in range(Jc):
        check_ind=np.where(MRdd[i:,j]==1)
        if len(check_ind[0]) !=0:
            if len(check_ind[0]) >1:
                elim_ind=np.delete(check_ind,0)
                MRdd[elim_ind+i,:]=np.mod(MRdd[elim_ind+i,:]+MRdd[check_ind[0][0]+i,:],2)
            MRdd[i,:],MRdd[check_ind[0][0]+i,:] = MRdd[check_ind[0][0]+i,:],MRdd[i,:].copy()
            i=i+1
    return i
############################


Lx,Ly=6,6
Lv=Lx*Ly
L=2*Lv
p_ind,v_ind=create_transformation_pv_link(Lv,Lx,Ly)
MR=initial_stabilizer_state(L,Lx,Ly,p_ind,v_ind)

#A_ind, B_ind, C_ind,AB_ind, AC_ind, BC_ind,ABC_ind=create_partition_TEE1(Lv,Lx,Ly,p_ind)

A_ind, B_ind, C_ind,AB_ind, AC_ind, BC_ind,ABC_ind=create_partition_TEE2(Lv,Lx,Ly,p_ind,v_ind)

TEE2=TEE_cal(MR,A_ind,B_ind,C_ind,AB_ind,AC_ind,BC_ind,ABC_ind)
#print(MR)
print("stop")
sys.exit()



A:ix= 0 iy= 2 ip= 12 [12, 48, 18, 49]
A:ix= 0 iy= 3 ip= 18 [18, 54, 24, 55]
A:ix= 0 iy= 4 ip= 24 [24, 60, 30, 61]
C:ix= 1 iy= 4 ip= 25 [25, 61, 31, 62]
C:ix= 2 iy= 4 ip= 26 [26, 62, 32, 63]
C:ix= 3 iy= 4 ip= 27 [27, 63, 33, 64]
B:ix= 1 iy= 2 ip= 13 [13, 49, 19, 50]
B:ix= 2 iy= 2 ip= 14 [14, 50, 20, 51]
B:ix= 3 iy= 2 ip= 15 [15, 51, 21, 52]
B:ix= 1 iy= 3 ip= 19 [19, 55, 25, 56]
B:ix= 2 iy= 3 ip= 20 [20, 56, 26, 57]
B:ix= 3 iy= 3 ip= 21 [21, 57, 27, 58]
TEE2
A_indd= [12, 18, 24, 30, 48, 49, 54, 55, 60, 61, 84, 90, 96, 102, 120, 121, 126, 127, 132, 133]
B_indd= [13, 14, 15, 19, 20, 21, 50, 51, 52, 56, 57, 58, 85, 86, 87, 91, 92, 93, 122, 123, 124, 128, 129, 130]
C_indd= [25, 26, 27, 31, 32, 33, 62, 63, 64, 97, 98, 99, 103, 104, 105, 134, 135, 136]
SA= 8 SB= 8 SC= 7 SAC= 13 SBC= 11 SAB= 14 SABC= 14
TEE= -1
stop


SystemExit: 

In [3]:
A = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

indices = [i for i, x in enumerate(A) if x == 1]
print(indices)

[15, 22, 62, 63]


In [17]:
from __future__ import print_function, division
import sys,os
# line 4 and line 5 below are for development purposes and ca be removed
qspin_path = os.path.join(os.getcwd(),"../../")
sys.path.insert(0,qspin_path)
from quspin.operators import hamiltonian # Hamiltonians and operators
#from quspin.basis import spinless_fermion_basis_1d # Hilbert space fermion basis
from quspin.basis import spin_basis_1d # Hilbert space spin basis
import numpy as np # generic math functions
import matplotlib.pyplot as plt # plotting library
from scipy.linalg import expm, sinm, cosm
from numpy.linalg import multi_dot
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy.sparse.linalg import inv, eigs
from scipy.linalg import svdvals
from scipy.stats import unitary_group
from scipy import linalg
from numpy import linalg as LA
import pylab
import time
import numba
from numba import njit,c16,i8,u2
import sys
import math

np.set_printoptions(precision=2)
np.set_printoptions(suppress=True)
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

time_sta = time.time()

def idx(x,y,Lx,Ly):
    return (x % Lx) + Lx * (y % Ly)
def inv_idx(i,Lx,Ly):
    return i % Lx, i // Lx


def dir_x(x,y,Lx,Lv):
    return x+Lx*y
def dir_y(x,y,Lx,Lv):
    return x+Lx*y+Lv
def dir_xy(x,y,Lx,Lv):
    return x+Lx*y+2*Lv


def create_transformation_pv_link(Lv, Lx, Ly):
    #############################################
    ############ plaquette part #################
    #############################################
    p_indd = []
    for ip in range(Lx * Ly):
        ix, iy = ip % Lx, ip // Lx
        ipx=(ix + 1) % Lx
        ipy=(iy + 1) % Ly
        p_indd.append([
            dir_x(ix, iy, Lx, Lv),
            dir_y(ix, iy, Lx, Lv),
            dir_x(ix, ipy, Lx, Lv),
            dir_y(ipx, iy, Lx, Lv)
        ])

    ############################################
    ############ vertex part ###################
    ############################################
    v_indd = []
    for iv in range(Lx * Ly):
        ix, iy = iv % Lx, iv // Lx
        imx=(ix - 1) % Lx
        imy=(iy - 1) % Ly
        v_indd.append([
            dir_x(ix, iy, Lx, Lv),
            dir_y(ix, iy, Lx, Lv),
            dir_x(imx, iy, Lx, Lv),
            dir_y(ix, imy, Lx, Lv),
        ])
    return p_indd, v_indd

#################################################################
def initial_stabilizer_state(Ld,Lxd,Lyd,p_ind,v_ind):
    # Ld: total # of qubits
    MRi=np.zeros((Ld,2*Ld+1),dtype='uint32')
    #print(2*Ld+1)
############################################
####### plaquette stabilizers ZZZZ #########
############################################
    for ip in range(Lxd*Lyd):
        MRi[ip][p_ind[ip]]=1 # neglect a phase factor
        #print(999,ip,p_ind[ip])
    #sys.exit()
############################################
###### vertex stabilizers  XXXX #############
############################################
    for iv in range((Lxd)*(Lyd)):
        ivs=iv+(Lxd*(Lyd))  # note that iv and ivs are different.
        MRi[ivs][v_ind[iv]]=1 # neglect a phase factor
        #print(ivs)
    #sys.exit()
    return MRi

def measurement_op(dMR,Ld,Lxd,Lyd,Gcd,mp,p_ind,v_ind):
    Meo=np.zeros(2*Ld+1,dtype='uint32')    
    if mp==1:
        ##Av measurement part
        vsite_list=np.arange((Lxd)*(Lyd)) #all vertex site
        vmd = np.random.choice(vsite_list, 1, replace=True)
        #print(vmd)
        Meo[v_ind[vmd[0]]]=1
    if mp==2:
        ##Bp measurement part
        psite_list=np.arange((Lxd)*(Lyd)) #all vertex site
        pmd = np.random.choice(psite_list, 1, replace=True)
        #print(pmd)
        Meo[p_ind[pmd[0]]]=1
    if mp==3:
        #local X
        site_list=np.arange(0,Ld-Lxd) #eliminate (low)smooth boundary
        md = np.random.choice(site_list, 1, replace=True)
        Meo[md]=1 #X
        # outcome 
        todo=[0,2]
        prob_list = [0.5,0.5]
        outcome = np.random.choice(todo,size=None,replace=True, p=prob_list)
        Meo[2*Ld] = outcome #factor +1 or -1 
    if mp==4:
        #local Z
        site_list=np.arange(Lxd,Ld) #eliminate (upper) rough boundary 
        md = np.random.choice(site_list, 1, replace=True)
        Meo[md+Ld]=1 #Z
        # outcome 
        todo=[0,2]
        prob_list = [0.5,0.5]
        outcome = np.random.choice(todo,size=None,replace=True, p=prob_list)
        Meo[2*Ld] = outcome #factor +1 or -1 
    
    # Meo without outcome sign
    MeoF=np.zeros(2*Ld,dtype='uint32')
    MeoF=Meo[0:2*Ld]
    
    # dMR without foctor
    dMRF=np.zeros((Ld,2*Ld),dtype='uint32')
    dMRF=dMR[:,0:2*Ld]
    #check anti commtation
    Mgs=np.dot(dMRF,np.dot(Gcd,MeoF.T))
    Mgs=Mgs%2  ## check: reduce Z2 value 
    
    antic_index_list=[]
    aid=np.where(Mgs[:]!=0)
    antic_index_list=aid[0]
    #######
    lenMe=len(antic_index_list)
    if lenMe !=0:
        kc=antic_index_list[0]
        dMR_prev=dMR[kc].copy()
        #replace stabilizer for kc
        dMR[kc]=Meo
        update_list=np.delete(antic_index_list,0)
        dMR[update_list,:]=np.mod(dMR[update_list,:]+dMR_prev,2)
        ## sigh_factorのupdateは今は何でもよい
    return dMR





#################################################
def create_partition_TEE(Ld,Lx,Ly,p_indd):
#############################################
############ plaquette part #################
#############################################
    A_indd=set()
    dhw=2
    dvw=2
    for iy in range(2,Ly-1,1):
        for ix in range(0,dhw,1):
            ip=ix+iy*Lx
            A_indd.update(p_indd[ip])
            
    B_indd=set()
    for iy in range(2,Ly-1,1):
        for ix in range(Lx-2-dhw,Lx-2,1):
            ip=ix+iy*Lx
            B_indd.update(p_indd[ip])
    
    C_indd=set()
    for iy in range(2,2+dvw,1):
        for ix in range(dhw,Lx-2-dhw,1):
            ip=ix+iy*Lx
            C_indd.update(p_indd[ip])    
    for iy in range(Ly-1-dvw,Ly-1,1):
        for ix in range(dhw,Lx-2-dhw,1):
            ip=ix+iy*Lx
            C_indd.update(p_indd[ip])
    C_indd=C_indd-A_indd-B_indd## elimination of elements included in A_indd and B_indd from C_indd
    AB_indd=A_indd.union(B_indd)
    AC_indd=A_indd.union(C_indd)
    BC_indd=B_indd.union(C_indd)
    ABC_indd=A_indd.union(B_indd,C_indd)
    A_indd=list(A_indd)
    A_indd+=[x-Ld for x in A_indd]
    print("A_indd=",A_indd)
    B_indd=list(B_indd)
    B_indd+=[x-Ld for x in B_indd]
    C_indd=list(C_indd)
    C_indd+=[x-Ld for x in C_indd]
    AB_indd=list(AB_indd)
    AB_indd+=[x-Ld for x in AB_indd]
    BC_indd=list(BC_indd)
    BC_indd+=[x-Ld for x in BC_indd]
    AC_indd=list(AC_indd)
    AC_indd+=[x-Ld for x in AC_indd]
    ABC_indd=list(ABC_indd)
    ABC_indd+=[x-Ld for x in ABC_indd]

    print("TEE1")
    print("A_indd=",A_indd)
    print("B_indd=",B_indd)
    print("C_indd=",C_indd)
    print("AB_indd=",AB_indd)
    print("AC_indd=",AC_indd)
    print("BC_indd=",BC_indd)
    print("ABC_indd=",ABC_indd)
    return A_indd, B_indd, C_indd,AB_indd, AC_indd, BC_indd,ABC_indd 
#####################################
#################################################
def create_partition_TEE2(Ld,Lx,Ly,p_indd):
#############################################
############ plaquette part #################
#############################################
    dhw=(Lx-2)//3
    Lv=Ly-3
    A_indd=set()
    for iy in range(2,Ly-1,1):
        for ix in range(0,dhw,1):
            ip=ix+iy*Lx
            A_indd.update(p_indd[ip])
    
    C_indd=set()
    for iy in range(2+Lv//2+1,Ly-1,1):
        for ix in range(dhw,Lx-2,1):
            ip=ix+iy*Lx
            C_indd.update(p_indd[ip])    
    C_indd=C_indd-A_indd

    B_indd=set()
    for iy in range(2,2+Lv//2+1,1):
        for ix in range(dhw,Lx-2,1):
            ip=ix+iy*Lx
            B_indd.update(p_indd[ip])    
    B_indd=B_indd-A_indd-C_indd
        
    C_indd=C_indd-A_indd-B_indd## elimination of elements included in A_indd and B_indd from C_indd
    AB_indd=A_indd.union(B_indd)
    AC_indd=A_indd.union(C_indd)
    BC_indd=B_indd.union(C_indd)
    ABC_indd=A_indd.union(B_indd,C_indd)
    A_indd=list(A_indd)
#    A_indd+=[x-Ld for x in A_indd]
    B_indd=list(B_indd)
#    B_indd+=[x-Ld for x in B_indd]
    C_indd=list(C_indd)
#    C_indd+=[x-Ld for x in C_indd]
    AB_indd=list(AB_indd)
#    AB_indd+=[x-Ld for x in AB_indd]
    BC_indd=list(BC_indd)
#    BC_indd+=[x-Ld for x in BC_indd]
    AC_indd=list(AC_indd)
#    AC_indd+=[x-Ld for x in AC_indd]
    ABC_indd=list(ABC_indd)
#    ABC_indd+=[x-Ld for x in ABC_indd]

    print("TEE2")
    print("A_indd=",A_indd)
    print("B_indd=",B_indd)
    print("C_indd=",C_indd)
    print("AB_indd=",AB_indd)
    print("AC_indd=",AC_indd)
    print("BC_indd=",BC_indd)
    print("ABC_indd=",ABC_indd)
    
    return A_indd, B_indd, C_indd,AB_indd, AC_indd, BC_indd,ABC_indd 
#####################################

def TEE_cal(MRFd,A_indd,B_indd,C_indd,AB_indd,AC_indd,BC_indd,ABC_indd):
    # partition AC#
    rankAC=rank_mod2_v3(MRFd[:,AC_indd].copy())
    SAC=rankAC-(len(AC_indd)//2)
######################
    rankAB=rank_mod2_v3(MRFd[:,AB_indd].copy())
    SAB=rankAB-(len(AB_indd)//2)
######################
    rankBC=rank_mod2_v3(MRFd[:,BC_indd].copy())
    SBC=rankBC-(len(BC_indd)//2)
######################
    rankABC=rank_mod2_v3(MRFd[:,ABC_indd].copy())
    SABC=rankABC-(len(ABC_indd)//2)
######################
    rankC=rank_mod2_v3(MRFd[:,C_indd].copy())
    SC=rankC-(len(C_indd)//2)
    rankB=rank_mod2_v3(MRFd[:,B_indd].copy())
    SB=rankB-(len(B_indd)//2)
    rankA=rank_mod2_v3(MRFd[:,A_indd].copy())
    SA=rankA-(len(A_indd)//2)
    TEE=SA+SB+SC-SAC-SBC-SAB+SABC
    print("SA=",SA,"SB=",SB,"SC=",SC,"SAC=",SAC,"SBC=",SBC,"SAB=",SAB,"SABC=",SABC)
    print("TEE=",TEE)
    return TEE

#######################################################################
def rank_mod2_v3(MRdd):
    # MRdA construct
    i = 0
    Ic = MRdd.shape[0]
    Jc = MRdd.shape[1]
    for j in range(Jc):
        check_ind=np.where(MRdd[i:,j]==1)
        if len(check_ind[0]) !=0:
            if len(check_ind[0]) >1:
                elim_ind=np.delete(check_ind,0)
                MRdd[elim_ind+i,:]=np.mod(MRdd[elim_ind+i,:]+MRdd[check_ind[0][0]+i,:],2)
            MRdd[i,:],MRdd[check_ind[0][0]+i,:] = MRdd[check_ind[0][0]+i,:],MRdd[i,:].copy()
            i=i+1
    return i
############################

#######################################################
#######################################################
#################     Main part   #####################
#######################################################
#######################################################
start=time.time()

Lx,Ly=7,7
L=2*Lx*Ly
Lv=Lx*Ly
p_ind, v_ind = create_transformation_pv_link(L,Lx,Ly)
###########
## TEE1 ###
###########
A_ind,B_ind,C_ind,AB_ind, AC_ind, BC_ind,ABC_ind=create_partition_TEE(L,Lx,Ly,p_ind)
###########
## TEE2 ###
###########
A2_ind,B2_ind,C2_ind,AB2_ind, AC2_ind, BC2_ind,ABC2_ind=create_partition_TEE2(L,Lx,Ly,p_ind)

print("stop")
sys.exit()



A_indd= [128, 133, 134, 135, 14, 15, 21, 22, 28, 29, 35, 36, 42, 43, 112, 113, 114, 119, 120, 121, 126, 127, 30, 35, 36, 37, -84, -83, -77, -76, -70, -69, -63, -62, -56, -55, 14, 15, 16, 21, 22, 23, 28, 29]
TEE1
A_indd= [128, 133, 134, 135, 14, 15, 21, 22, 28, 29, 35, 36, 42, 43, 112, 113, 114, 119, 120, 121, 126, 127, 30, 35, 36, 37, -84, -83, -77, -76, -70, -69, -63, -62, -56, -55, 14, 15, 16, 21, 22, 23, 28, 29]
B_indd= [129, 130, 131, 136, 137, 138, 17, 18, 24, 25, 31, 32, 38, 39, 45, 46, 115, 116, 117, 122, 123, 124, 31, 32, 33, 38, 39, 40, -81, -80, -74, -73, -67, -66, -60, -59, -53, -52, 17, 18, 19, 24, 25, 26]
C_indd= [37, 44, 16, 23, 30, -61, -54, -82, -75, -68]
AB_indd= [128, 129, 130, 131, 133, 134, 135, 136, 137, 138, 14, 15, 17, 18, 21, 22, 24, 25, 28, 29, 31, 32, 35, 36, 38, 39, 42, 43, 45, 46, 112, 113, 114, 115, 116, 117, 119, 120, 121, 122, 123, 124, 126, 127, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, -84, -83, -81, -80, -77, -76, -74, -73, -70, -69, -67, -66, -63, -62, 

SystemExit: 